In [0]:
from pyspark.sql.functions import *

# Section 1: Data Quality Summary

## Total source count

In [0]:
display(
    spark.sql("""
    SELECT COUNT(*) AS source_count
    FROM grocery_delivery_project.bronze.grocery_source
    """)
)

## Rejected Records

In [0]:
display(
    spark.sql("""
    SELECT COUNT(*) AS rejected_count
    FROM grocery_delivery_project.bronze.bad_records
    """)
)

## Duplicate count

In [0]:
duplicate=display(
    spark.sql("""
    SELECT
        SUM(record_count - 1) AS duplicate_count
    FROM (
        SELECT
            order_id,
            COUNT(*) AS record_count
        FROM grocery_delivery_project.bronze.grocery_with_duplicate
        GROUP BY order_id
        HAVING COUNT(*) > 1
    ) d
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        SUM(record_count - 1) AS duplicate_count
    FROM (
        SELECT
            order_id,
            COUNT(*) AS record_count
        FROM grocery_delivery_project.bronze.grocery_with_duplicate
        GROUP BY order_id
        HAVING COUNT(*) > 1
    ) d
    """)
)

## Clean Records

In [0]:
display(
    spark.sql("""
    SELECT COUNT(*) AS clean_records
    FROM grocery_delivery_project.silver.silver_table
    """)
)

## Data Reconciliation

In [0]:
display(
    spark.sql("""
    SELECT
        (SELECT COUNT(*) 
         FROM grocery_delivery_project.bronze.grocery_source) AS source_count,

        (SELECT COUNT(*) 
         FROM grocery_delivery_project.bronze.bad_records) AS rejected_count,

        (SELECT COALESCE(SUM(record_count - 1),0)
         FROM (
             SELECT
                 order_id,
                 COUNT(*) AS record_count
             FROM grocery_delivery_project.bronze.grocery_with_duplicate
             GROUP BY order_id
             HAVING COUNT(*) > 1
         ) d) AS duplicate_record_count,

        (SELECT COUNT(*) 
         FROM grocery_delivery_project.silver.silver_table) AS loaded_count
    """)
)

## Reject Reasons And its Count

In [0]:
display(
    spark.sql("""
    SELECT 'Missing Order ID' AS rejection_reason, COUNT(*) AS rejected_count
    FROM grocery_delivery_project.bronze.bad_records
    WHERE order_id IS NULL

    UNION ALL

    SELECT 'Missing Customer Name', COUNT(*)
    FROM grocery_delivery_project.bronze.bad_records
    WHERE customer_name IS NULL

    UNION ALL

    SELECT 'Missing Category', COUNT(*)
    FROM grocery_delivery_project.bronze.bad_records
    WHERE category IS NULL

    UNION ALL

    SELECT 'Invalid Quantity', COUNT(*)
    FROM grocery_delivery_project.bronze.bad_records
    WHERE quantity <= 0

    UNION ALL

    SELECT 'Invalid Unit Price', COUNT(*)
    FROM grocery_delivery_project.bronze.bad_records
    WHERE unit_price <= 0

    UNION ALL

    SELECT 'Invalid Discount', COUNT(*)
    FROM grocery_delivery_project.bronze.bad_records
    WHERE discount < 0

    UNION ALL

    SELECT 'Invalid Delivery Status', COUNT(*)
    FROM grocery_delivery_project.bronze.bad_records
    WHERE delivery_status = 'INVALID_STATUS'
    """)
)

# Business KPI Dashboard

### 1. Total Revenue KPI

In [0]:
display(
    spark.sql("""
    SELECT
        ROUND(SUM(net_sales),2) AS total_revenue
    FROM grocery_delivery_project.gold.fact_orders
    """)
)

### 2. Category-wise Sales

In [0]:
display(
    spark.sql("""
    SELECT
        c.category,
        SUM(f.net_sales) AS total_sales
    FROM grocery_delivery_project.gold.fact_orders f
    JOIN grocery_delivery_project.gold.dim_category c
        ON f.category_sk = c.category_sk
    GROUP BY c.category
    ORDER BY total_sales DESC
    """)
)

### 3.City-wise Sales

In [0]:
display(
    spark.sql("""
    SELECT
        c.city,
        SUM(f.net_sales) AS total_sales
    FROM grocery_delivery_project.gold.fact_orders f
    JOIN grocery_delivery_project.gold.dim_city c
        ON f.city_sk = c.city_sk
    GROUP BY c.city
    ORDER BY total_sales DESC
    """)
)

### 4.Delivery Status KPI

In [0]:
display(
    spark.sql("""
    SELECT
        delivery_status,
        COUNT(*) AS total_orders
    FROM grocery_delivery_project.gold.fact_orders
    GROUP BY delivery_status
    """)
)

### Sales Trend by Month

In [0]:
display(
    spark.sql("""
    SELECT
        d.year,
        d.month,
        d.month_name,
        SUM(f.net_sales) AS total_sales
    FROM grocery_delivery_project.gold.fact_orders f
    JOIN grocery_delivery_project.gold.dim_date d
        ON f.date_sk = d.date_sk
    GROUP BY
        d.year,
        d.month,
        d.month_name
    ORDER BY
        d.year,
        d.month
    """)
)